In [ ]:
# import numpy as np

# # Function
# def f(x):
#     return np.exp(-x**2)

# # Trapezoid rule
# def trapezoid(f, a, b, n):
#     x = np.linspace(a, b, n+1)
#     y = f(x)
#     dx = (b - a)/n
#     return (dx/2) * np.sum(y[:-1] + y[1:])

# # Adaptive trapezoid
# def adaptive_trapezoid(f, a, b, tol=1e-10):
#     n = 2
#     prev = trapezoid(f, a, b, n)
#     steps = [(n, prev)]
#     while True:
#         n *= 2
#         curr = trapezoid(f, a, b, n)
#         steps.append((n, curr))
#         if abs(curr - prev) < tol:
#             return curr, steps
#         prev = curr

# # Parameters
# a, b = 0, 1
# I_exact = 0.746824132812  # reference value

# # Fixed trapezoid results
# n_values = [2, 4, 8, 16, 32, 64, 128, 256]
# print("=== Trapezoid Rule (Fixed n) ===")
# for n in n_values:
#     I_n = trapezoid(f, a, b, n)
#     error = abs(I_n - I_exact)
#     print(f"n={n:4d} | Result={I_n:.12f} | Error={error:.2e}")

# # Adaptive trapezoid results
# print("\n=== Adaptive Trapezoid Refinement ===")
# I_adapt, steps = adaptive_trapezoid(f, a, b, tol=1e-10)
# for n, val in steps:
#     error = abs(val - I_exact)
#     print(f"n={n:4d} | Result={val:.12f} | Error={error:.2e}")

# print(f"\nFinal Adaptive Result = {I_adapt:.12f}")
# print(f"Exact Analytical Value = {I_exact:.12f}")


In [6]:
import ROOT
import numpy as np

def RosenBrock(vecx):
    x = vecx[0]
    y = vecx[1]
    return (1 - x)**2 + 100 * (y - x**2)**2  # Correct Rosenbrock function

# create minimizer giving a name and a name (optionally) for the specific algorithm
#  possible choices are:
#     minimizerName                  algoName
#
#     Minuit                     Migrad, Simplex,Combined,Scan  (default is Migrad)
#     Minuit2                    Migrad, BFGS, Simplex,Combined,Scan  (default is Migrad)
#     GSLMultiMin                ConjugateFR, ConjugatePR, BFGS, BFGS2, SteepestDescent
#     GSLSimAn
#     Genetic

def NumericalMinimization(minimizerName="Minuit2",
                          algoName="Migrad",
                          randomSeed=-1):
    
    minimizer = ROOT.Math.Factory.CreateMinimizer(minimizerName, algoName)
    if (not minimizer):
        raise RuntimeError(
            "Cannot create minimizer \"{}\". Maybe the required library was not built?".format(minimizerName))

    # Set tolerance and other minimizer parameters, one can also use default
    # values

    minimizer.SetMaxFunctionCalls(1000000)  # working for Minuit/Minuit2
    # for GSL minimizers - no effect in Minuit/Minuit2
    minimizer.SetMaxIterations(10000)
    minimizer.SetTolerance(0.001)
    minimizer.SetPrintLevel(1)

    # Create function wrapper for minimizer

    f = ROOT.Math.Functor(RosenBrock, 2)

    # Starting point
    variable = [-1., 1.2]
    step = [0.01, 0.01]
    if (randomSeed >= 0):
        r = ROOT.TRandom2(randomSeed)
        variable[0] = r.Uniform(-20, 20)
        variable[1] = r.Uniform(-20, 20)

    minimizer.SetFunction(f)

    # Set the free variables to be minimized !
    minimizer.SetVariable(0, "x", variable[0], step[0])
    minimizer.SetVariable(1, "y", variable[1], step[1])

    # Do the minimization
    ret = minimizer.Minimize()

    xs = minimizer.X()
    print("Minimum: f({} , {}) = {}".format(xs[0], xs[1], minimizer.MinValue()))

    # Expected minimum is f(1,1) = 0
    expected_min = 0.0
    tolerance = 1.E-4
    if (ret and abs(minimizer.MinValue() - expected_min) < tolerance):
        print("Minimizer {} - {} converged to the right minimum!".format(minimizerName, algoName))
    else:
        print("Minimizer {} - {} failed to converge! Found minimum at f({}, {}) = {}".format(
            minimizerName, algoName, xs[0], xs[1], minimizer.MinValue()))
        # Don't raise an error for demonstration, just print warning

if __name__ == "__main__":
    NumericalMinimization()

Minimum: f(0.9998980090368492 , 0.9997861235836393) = 2.0212845544987252e-08
Minimizer Minuit2 - Migrad converged to the right minimum!
Minuit2Minimizer: Minimize with max-calls 1000000 convergence for edm < 0.001 strategy 1
Minuit2Minimizer : Valid minimum - status = 0
FVAL  = 2.02128455449872524e-08
Edm   = 2.02984116502312438e-08
Nfcn  = 174
x	  = 0.999898	 +/-  1.00391
y	  = 0.999786	 +/-  2.01014


In [ ]:
# import numpy as np
# from scipy.optimize import curve_fit
# from scipy.stats import t

# # Example model
# def model(x, a, b, c, d):
#     return a * np.exp(-b * x) + c * x + d

# # Fake data
# xdata = np.linspace(0, 4, 50)
# y = model(xdata, 2.5, 1.3, 0.5, 1.0)
# rng = np.random.default_rng(0)
# y_noise = y + 0.2 * rng.normal(size=len(xdata))

# # Fit
# popt, pcov = curve_fit(model, xdata, y_noise)

# # Extract standard errors
# perr = np.sqrt(np.diag(pcov))

# # 90% confidence intervals
# alpha = 0.10
# dof = max(0, len(xdata) - len(popt))  # degrees of freedom
# tval = t.ppf(1.0 - alpha/2., dof)

# ci = [ (p - tval*e, p + tval*e) for p, e in zip(popt, perr) ]

# print("Parameters:", popt)
# print("90% CI:", ci)


In [ ]:
# import numpy as np
# from iminuit import Minuit
# from iminuit.cost import LeastSquares

# # Example model
# def model(x, a, b, c, d):
#     return a * np.exp(-b * x) + c * x + d

# # Fake data
# x = np.linspace(0, 4, 50)
# y = model(x, 2.5, 1.3, 0.5, 1.0) + 0.2 * np.random.normal(size=len(x))
# yerr = np.full_like(y, 0.2)

# # Least squares cost
# cost = LeastSquares(x, y, yerr, model)
# m = Minuit(cost, a=1, b=1, c=1, d=1)  # initial guesses
# m.migrad()  # minimize
# m.hesse()   # covariance matrix

# print(m.values)   # best-fit params
# print(m.errors)   # 1σ errors (~68%)

# # For 90% CI, use m.mnprofile or m.mncontour
# for name in m.parameters:
#     ci90 = m.draw_mnprofile(name)  # 90% ≈ 1.64σ
#     # print(name, "90% CI:", ci90)


In [ ]:
# import ROOT
# import numpy as np

# # Sample data for chi-squared calculation
# # You can replace this with your actual data
# x_data = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
# y_data = np.array([2.1, 3.9, 6.1, 8.0, 9.9])
# y_errors = np.array([0.2, 0.3, 0.2, 0.4, 0.3])

# def linear_model(x, params):
#     """Linear model: y = a*x + b"""
#     a, b = params[0], params[1]
#     return a * x + b

# def chi2_function(params):
#     """
#     Calculate chi-squared for a linear fit
#     params[0] = slope (a)
#     params[1] = intercept (b)
#     """
#     chi2 = 0.0
#     for i in range(len(x_data)):
#         predicted = linear_model(x_data[i], params)
#         residual = y_data[i] - predicted
#         chi2 += (residual / y_errors[i])**2
#     return chi2

# # Alternative: Generic chi2 function for any model
# def generic_chi2_function(params):
#     """
#     Generic chi-squared function - modify this for your specific model
#     """
#     # Example: quadratic model y = a*x^2 + b*x + c
#     # Uncomment and modify as needed:
    
#     # chi2 = 0.0
#     # for i in range(len(x_data)):
#     #     predicted = params[0]*x_data[i]**2 + params[1]*x_data[i] + params[2]
#     #     residual = y_data[i] - predicted
#     #     chi2 += (residual / y_errors[i])**2
#     # return chi2
    
#     # For now, use the linear model
#     return chi2_function(params)

# def NumericalMinimization(minimizerName="Minuit2",
#                           algoName="",
#                           randomSeed=-1):
    
#     minimizer = ROOT.Math.Factory.CreateMinimizer(minimizerName, algoName)
#     if (not minimizer):
#         raise RuntimeError(
#             "Cannot create minimizer \"{}\". Maybe the required library was not built?".format(minimizerName))
    
#     # Set tolerance and other minimizer parameters
#     minimizer.SetMaxFunctionCalls(1000000)
#     minimizer.SetMaxIterations(10000)
#     minimizer.SetTolerance(0.001)
#     minimizer.SetPrintLevel(1)
    
#     # Create function wrapper for minimizer
#     # Use 2 parameters for linear fit (slope and intercept)
#     f = ROOT.Math.Functor(chi2_function, 2)
    
#     # Starting point for parameters [slope, intercept]
#     # You can estimate these from your data or use reasonable guesses
#     variable = [1.0, 0.0]  # Initial guess: slope=1, intercept=0
#     step = [0.01, 0.01]    # Step sizes for parameters
    
#     if (randomSeed >= 0):
#         r = ROOT.TRandom2(randomSeed)
#         variable[0] = r.Uniform(-10, 10)  # Random slope
#         variable[1] = r.Uniform(-10, 10)  # Random intercept
    
#     minimizer.SetFunction(f)
    
#     # Set the free variables to be minimized
#     minimizer.SetVariable(0, "slope", variable[0], step[0])
#     minimizer.SetVariable(1, "intercept", variable[1], step[1])
    
#     # Optional: Set parameter limits if needed
#     # minimizer.SetVariableLimits(0, -100, 100)  # Limit slope
#     # minimizer.SetVariableLimits(1, -100, 100)  # Limit intercept
    
#     # Do the minimization
#     ret = minimizer.Minimize()
    
#     xs = minimizer.X()
#     chi2_min = minimizer.MinValue()
    
#     print("Minimum: chi2(slope={:.4f}, intercept={:.4f}) = {:.4f}".format(
#         xs[0], xs[1], chi2_min))
    
#     # Calculate degrees of freedom
#     ndf = len(x_data) - 2  # n_data_points - n_parameters
#     reduced_chi2 = chi2_min / ndf
    
#     print("Reduced chi2 = {:.4f}".format(reduced_chi2))
#     print("Number of degrees of freedom = {}".format(ndf))
    
#     # Check convergence
#     if ret:
#         print("Minimizer {} - {} converged successfully!".format(minimizerName, algoName))
        
#         # Print parameter errors if available
#         if minimizer.Errors():
#             errors = minimizer.Errors()
#             print("Parameter errors:")
#             print("  slope error = {:.4f}".format(errors[0]))
#             print("  intercept error = {:.4f}".format(errors[1]))
#     else:
#         print("Minimizer {} - {} failed to converge !!!".format(minimizerName, algoName))
#         raise RuntimeError("NumericalMinimization failed to converge!")
    
#     return xs, chi2_min, reduced_chi2

# def plot_results(params):
#     """Optional: Plot the data and fitted curve"""
#     try:
#         import matplotlib.pyplot as plt
        
#         # Generate points for the fitted line
#         x_fit = np.linspace(min(x_data), max(x_data), 100)
#         y_fit = linear_model(x_fit, params)
        
#         plt.figure(figsize=(8, 6))
#         plt.errorbar(x_data, y_data, yerr=y_errors, fmt='o', label='Data')
#         plt.plot(x_fit, y_fit, 'r-', label=f'Fit: y = {params[0]:.3f}x + {params[1]:.3f}')
#         plt.xlabel('x')
#         plt.ylabel('y')
#         plt.legend()
#         plt.grid(True)
#         plt.title('Chi-squared Fit Results')
#         plt.show()
        
#     except ImportError:
#         print("matplotlib not available for plotting")

# if __name__ == "__main__":
#     # Run the minimization
#     fitted_params, min_chi2, reduced_chi2 = NumericalMinimization()
    
#     # Optionally plot results
#     # plot_results(fitted_params)